In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create figures folder automatically
os.makedirs("figures", exist_ok=True)

# Load clean dataset
df = pd.read_csv("telco_churn_clean.csv")

# --- 1. PANDAS OPERATIONS (groupby, agg, pivot_table, transform, query, sort_values) ---
overall_churn = df["Churn_Flag"].mean() * 100
print(f"1. Overall Churn Rate: {overall_churn:.2f}% ({df['Churn_Flag'].sum()} of {len(df)} customers)")

contract_summary = (
    df.groupby(["Contract", "InternetService"], as_index=False)
      .agg(
          customers=("customerID", "count"),
          churn_rate=("Churn_Flag", "mean"),
          avg_monthly=("MonthlyCharges", "mean"),
          median_tenure=("tenure", "median")
      )
      .sort_values(by="churn_rate", ascending=False)
)
print("\n2. Top Churn Segments (Contract x InternetService):")
print(contract_summary.head(5))

pivot_churn = pd.pivot_table(
    df,
    values="Churn_Flag",
    index="Tenure_Band",
    columns="Contract",
    aggfunc="mean",
    observed=False
)

# --- 2. NUMPY OPERATIONS (percentiles, IQR, bootstrap CI, vectorised risk bands, simulation) ---
churned_charges = df.loc[df["Churn_Flag"] == 1, "MonthlyCharges"].to_numpy()
retained_charges = df.loc[df["Churn_Flag"] == 0, "MonthlyCharges"].to_numpy()

q25_c, med_c, q75_c = np.percentile(churned_charges, [25, 50, 75])
q25_r, med_r, q75_r = np.percentile(retained_charges, [25, 50, 75])
print(f"\n3. Median MonthlyCharges -> Churned: ${med_c:.2f} (IQR: ${q75_c-q25_c:.2f}) | Retained: ${med_r:.2f} (IQR: ${q75_r-q25_r:.2f})")

rng = np.random.default_rng(42)
boot_diffs = np.mean(rng.choice(churned_charges, (5000, len(churned_charges)), replace=True), axis=1) - \
             np.mean(rng.choice(retained_charges, (5000, len(retained_charges)), replace=True), axis=1)
ci_low, ci_high = np.quantile(boot_diffs, [0.025, 0.975])
print(f"4. Mean MonthlyCharges Difference: +${np.mean(boot_diffs):.2f} (95% CI: [${ci_low:.2f}, ${ci_high:.2f}])")

# Risk Band Segmentation
conditions = [
    (df["Contract"] == "Month-to-month") & (df["tenure"] <= 12) & (df["MonthlyCharges"] > 70),
    (df["Contract"] == "Month-to-month") & (df["tenure"] <= 24),
    (df["Contract"] == "One year")
]
choices = ["High Risk", "Elevated Risk", "Moderate Risk"]
df["Risk_Band"] = np.select(conditions, choices, default="Low Risk")

risk_summary = df.groupby("Risk_Band", as_index=False).agg(
    customers=("customerID", "count"),
    churn_rate=("Churn_Flag", "mean"),
    avg_monthly=("MonthlyCharges", "mean")
).sort_values("churn_rate", ascending=False)
print("\n5. Risk Band Breakdown:")
print(risk_summary)

# Retention Intervention Simulation
high_risk = df.query("Risk_Band == 'High Risk'")
saved_customers = len(high_risk) * 0.15 * (high_risk["Churn_Flag"].mean() * 0.50)
annual_rev_saved = saved_customers * high_risk["MonthlyCharges"].mean() * 12
print(f"\n6. Simulation: Converting 15% of High Risk customers to 1-yr contracts saves ~{saved_customers:.0f} customers (${annual_rev_saved:,.2f}/yr)")

# --- 3. GENERATE & SAVE ALL 5 REQUIRED CHARTS ---
plt.rcParams["figure.dpi"] = 120

# Chart 1: Churn Rate by Contract Type
fig, ax = plt.subplots(figsize=(7, 4))
c_rates = df.groupby("Contract")["Churn_Flag"].mean().sort_values(ascending=False) * 100
bars = ax.bar(c_rates.index, c_rates.values, color=["#d9534f", "#f0ad4e", "#5cb85c"])
ax.set_title("Churn Rate by Contract Type (%)")
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, 55)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2, f"{bar.get_height():.1f}%", ha="center")
plt.tight_layout()
plt.savefig("figures/01_churn_by_contract.png")
plt.close()

# Chart 2: Heatmap of Tenure Band vs Contract
fig, ax = plt.subplots(figsize=(7, 4.5))
order_bands = ["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]
order_contracts = ["Month-to-month", "One year", "Two year"]
hm_data = pivot_churn.reindex(index=order_bands, columns=order_contracts) * 100
im = ax.imshow(hm_data.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(order_contracts)))
ax.set_xticklabels(order_contracts)
ax.set_yticks(range(len(order_bands)))
ax.set_yticklabels(order_bands)
ax.set_title("Churn Rate (%) by Tenure Band and Contract")
for i in range(len(order_bands)):
    for j in range(len(order_contracts)):
        val = hm_data.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center", color="black")
plt.colorbar(im, ax=ax, label="Churn Rate (%)")
plt.tight_layout()
plt.savefig("figures/02_tenure_contract_heatmap.png")
plt.close()

# Chart 3: Monthly Charges Distribution (Retained vs Churned)
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([retained_charges, churned_charges], tick_labels=["Retained", "Churned"], patch_artist=True)
ax.set_title("Monthly Charges Distribution by Churn Status ($)")
ax.set_ylabel("Monthly Charges ($ USD)")
plt.tight_layout()
plt.savefig("figures/03_monthly_charges_boxplot.png")
plt.close()

# Chart 4: Churn Rate by Payment Method
fig, ax = plt.subplots(figsize=(8, 4))
p_rates = df.groupby("PaymentMethod")["Churn_Flag"].mean().sort_values() * 100
bars = ax.barh(p_rates.index, p_rates.values, color="#4a90e2")
ax.set_title("Churn Rate by Payment Method (%)")
ax.set_xlabel("Churn Rate (%)")
ax.set_xlim(0, 55)
for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f"{bar.get_width():.1f}%", va="center")
plt.tight_layout()
plt.savefig("figures/04_churn_by_payment_method.png")
plt.close()

# Chart 5: Customer Volume & Churn Rate by Risk Band
fig, ax1 = plt.subplots(figsize=(7.5, 4.5))
band_order = ["High Risk", "Elevated Risk", "Moderate Risk", "Low Risk"]
rb = risk_summary.set_index("Risk_Band").reindex(band_order)
ax1.bar(rb.index, rb["customers"], color="#8faadc", label="Customer Count")
ax1.set_ylabel("Number of Customers", color="#2f5597")
ax2 = ax1.twinx()
ax2.plot(rb.index, rb["churn_rate"] * 100, color="#c00000", marker="o", linewidth=2, label="Churn Rate (%)")
ax2.set_ylabel("Churn Rate (%)", color="#c00000")
ax2.set_ylim(0, 75)
ax1.set_title("Customer Volume & Churn Rate Across Risk Tiers")
plt.tight_layout()
plt.savefig("figures/05_risk_band_performance.png")
plt.close()

print("\nAll 5 charts saved to figures/ successfully!")

1. Overall Churn Rate: 26.54% (1869 of 7043 customers)

2. Top Churn Segments (Contract x InternetService):
         Contract InternetService  customers  churn_rate  avg_monthly  \
1  Month-to-month     Fiber optic       2128    0.546053    87.021194   
0  Month-to-month             DSL       1223    0.322159    50.219501   
4        One year     Fiber optic        539    0.192950    98.779499   
2  Month-to-month              No        524    0.188931    20.409542   
3        One year             DSL        570    0.092982    61.396754   

   median_tenure  
1           16.0  
0           10.0  
4           55.0  
2            5.0  
3           41.0  

3. Median MonthlyCharges -> Churned: $79.65 (IQR: $38.05) | Retained: $64.43 (IQR: $63.30)
4. Mean MonthlyCharges Difference: +$13.18 (95% CI: [$11.75, $14.58])

5. Risk Band Breakdown:
       Risk_Band  customers  churn_rate  avg_monthly
1      High Risk        856    0.690421    83.084871
0  Elevated Risk       1875    0.379200    51.